In [ ]:
import polars as pl 
#import ollama
import asyncio
import tqdm
import pickle
import time
import os
import ollama
from datasets import Dataset
from transformers import pipeline, Pipeline, AutoTokenizer
from transformers.pipelines.pt_utils import KeyDataset
import emojis
import torch
import numpy as np

# Dla szybkiego pobierania modelu
os.environ["HF_XET_HIGH_PERFORMANCE"]="1"
emotions_all_alt = ["gratitude", "sadness", "anger", "excitement", "admiration", "confusion", "neutral"]
emotions_all = [
    "joy",
    "anger",
    "sadness",
    "sarcasm",
    "curiosity",
    "gratitude",
    "neutral"
]


/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [89]:
data = pl.read_csv("hf://datasets/AmaanP314/youtube-comment-sentiment/youtube-comments-sentiment.csv")
data = (data
        .with_columns(Language = pl.read_csv("../src/src_files/data/language.csv").to_series())
        .filter(
            (pl.col("Language") == "en").and_(
                pl.col("AuthorChannelID") != "AugmentedCID"
            )
        ))

In [4]:
comments = (data
            .select(
                pl.col("CommentText")
                .map_elements(emojis.decode, return_dtype=pl.String)
                .str.replace_all("</s>", "")
            )
            )
hg_data = Dataset.from_polars(comments)

In [16]:
comments.write_csv("training.csv")

In [2]:
comments = pl.read_csv("training.csv")
hg_data = Dataset.from_polars(comments)

In [3]:
class Labeller:
    def __init__(self,
                 model_name : str):
        self.begin = 0
        self.labels: list[dict[str, float]] = []
        self.model_name = model_name

    @classmethod
    def restore(cls,
                model_name : str
                ):
        try:
            with open(f'checkpoints/ckpt_{model_name}.pkl', "rb") as f:
                obj = pickle.load(f)
            print(f"Loaded for step: {obj.begin}")
            return obj
        except FileNotFoundError:
            print("File not found, starting from the begginng") 
            return cls(model_name)  


    def label(self,
              texts: Dataset,
              checkpoint_threshold: int,
              pipe: Pipeline
              ) -> None:
        
        texts = texts.select(range(self.begin, len(texts)))
        
        for out in tqdm.tqdm(
                pipe(
                    KeyDataset(texts, "CommentText"),
                    candidate_labels=emotions_all,
                    batch_size=16, truncation=True, add_special_tokens=True, max_length=300
                ), initial=self.begin, total = len(texts) + self.begin
            ):

            self.labels.append(
                dict(zip(out["labels"], out["scores"]))
            )
            self.begin += 1

            if self.begin % checkpoint_threshold == 0:
                
                with open(f'checkpoints/ckpt_{self.model_name}.pkl', "wb+") as f:
                    pickle.dump(self, f)
                    
        with open(f'checkpoints/ckpt_{self.model_name}.pkl', "wb+") as f:
                    pickle.dump(self, f)

## BART 
[model_card](https://huggingface.co/facebook/bart-large-mnli)

In [2]:
pipe = pipeline(model="facebook/bart-large-mnli", dtype='float16')

Device set to use cuda:0
/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU1 NVIDIA GeForce GTX 1050 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce GTX 1050 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to us

In [55]:
lab = Labeller.restore("bart")
lab.label(
    hg_data,
    10_000,
    pipe
)

Loaded for step: 700000


100%|██████████| 790369/790369 [32:18<00:00, 46.62it/s]  


In [57]:
with open(f'checkpoints/ckpt_bart.pkl', "wb+") as f:
                    pickle.dump(lab, f)

## deberta
[model_card](https://huggingface.co/MoritzLaurer/DeBERTa-v3-base-mnli)

In [4]:
deberta = pipeline(task =  "zero-shot-classification",model ="MoritzLaurer/DeBERTa-v3-base-mnli", use_fast=False)

Device set to use cuda:0
/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU1 NVIDIA GeForce GTX 1050 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/jakub/python/zaliczenie/WizZali/yt_zal/lib/python3.12/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce GTX 1050 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to us

In [ ]:
deberta_lab = Labeller.restore("deberta")

Loaded for step: 790369


In [6]:
deberta_lab.label(
    hg_data,
    50_000,
    deberta
)

100%|██████████| 790369/790369 [1:09:38<00:00, 45.56it/s] 


In [7]:
len(deberta_lab.labels)

790369

# Evaluation

In [27]:
bart_lab = Labeller.restore("bart")
deberta_lab = Labeller.restore("deberta")

Loaded for step: 790369
Loaded for step: 790369


In [30]:
top_bart = [max(p, key=p.get) for p in bart_lab.labels]
top_deberta = [max(p, key=p.get) for p in deberta_lab.labels]

In [66]:
def entropy (
             x: dict[str, float]
            ) -> float:
    h = 0
    for p in x.values():
        h += -p * np.log(p) / np.log(7)
    
    return h


In [31]:
emotions = pl.DataFrame(
    {
        "Bart"    : pl.Series(top_bart),
        "Deberta" : pl.Series(top_deberta)
    }
)

In [55]:
(
    emotions["Bart"]
    .value_counts(sort=True)
    .rename({
        "Bart":"Emotion",
        "count":"Bart"
    })
    .join(
        emotions["Deberta"].value_counts().rename({"Deberta":"Emotion", "count":"Deberta"}),
        on="Emotion")
)

Emotion,Bart,Deberta
str,u32,u32
"""sarcasm""",234374,80277
"""gratitude""",100569,124911
"""joy""",168574,134039
"""sadness""",45468,49697
"""anger""",58248,41140
"""neutral""",32164,34550
"""curiosity""",150972,325755


In [63]:
(emotions
 .select(
     (pl.col("Bart").ne(pl.col("Deberta")).sum() / len(emotions) * 100).alias("Pct of different emotions")
 )
)

Pct of different emotions
f64
46.299766


In [57]:
emotions.head()

Bart,Deberta
str,str
"""curiosity""","""curiosity"""
"""joy""","""joy"""
"""curiosity""","""curiosity"""
"""curiosity""","""gratitude"""
"""gratitude""","""neutral"""


In [67]:
entropies_bart = list(map(entropy, bart_lab.labels))
confidence_bart = list(map(lambda x: max(x.values()), bart_lab.labels))
entropies_deberta= list(map(entropy, deberta_lab.labels))
confidence_deberta = list(map(lambda x: max(x.values()), deberta_lab.labels))

In [ ]:
(emotions
 .with_columns(
     Entropy_BART = pl.Series(entropies_bart),
     Confidence_BART = pl.Series(confidence_bart),
     Entropy_DEBERTA = pl.Series(entropies_deberta),
     Confidence_DEBERTA = pl.Series(confidence_deberta)
 )
 .select(
     pl.col("Entropy_BART").mean(),
     pl.col("Entropy_DEBERTA").mean(),
     pl.col("Confidence_BART").mean(),
     pl.col("Confidence_DEBERTA").mean()
 )
 )

Entropy_BART,Entropy_DEBERTA,Confidence_BART,Confidence_DEBERTA
f64,f64,f64,f64
0.685937,0.669667,0.510951,0.530039


## Combining propabilities

In [81]:
combined: list[dict[str, float]] = []

for bart, deberta in zip(bart_lab.labels, deberta_lab.labels):
    p = {emotion: (bart[emotion] + deberta[emotion])/2 for emotion in emotions_all}
    combined.append(p)

In [86]:
top_combined = [max(p, key=p.get) for p in combined]

In [88]:
pl.Series(top_combined).value_counts(sort=True)

,count
str,u32
"""curiosity""",238859
"""joy""",160087
"""sarcasm""",151706
"""gratitude""",111003
"""sadness""",49320
"""anger""",47793
"""neutral""",31601


In [90]:
(data
 .with_columns(
     Emotions = pl.Series(top_combined)
 )
 .write_csv("comments_emotions.csv"))